In [1]:
import pandas as pd
import sys
sys.path.append("../src/process_results")

from tqdm.auto import tqdm
tqdm.pandas()

from utils import fix_exists_by_status

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('../data/results_gpt.csv')
df.shape

(2135, 4)

In [4]:
df_ws = pd.read_csv('../data/results_gpt_web_search.csv')
df_ws.shape

(1042, 4)

In [3]:
df_ws2 = pd.read_csv('../data/results_pend_gpt.csv')
df_ws2.shape

(116, 4)

In [13]:
df_ws.references.isna().sum(), df_ws[df_ws.references == '[]'].shape[0]

(np.int64(0), 1)

In [10]:
df_ws2.loc[(df_ws2.references.isna()) | (df_ws2.references == '[]')].result.iloc[0]


'Data centers utilize water both directly for liquid cooling and indirectly for electricity generation. Liquid cooling systems employ water to dissipate heat from servers, enhancing efficiency and performance. Additionally, water is integral to power plants supplying electricity to data centers, as it is used in cooling processes.\n\n**Strength of Evidence:**\n\n- **Direct Water Use in Liquid Cooling:** Studies indicate that liquid cooling systems in data centers can reduce energy consumption by up to 50% compared to traditional air cooling methods. This substantial reduction underscores the effectiveness of water-based cooling solutions.\n\n- **Indirect Water Use in Electricity Generation:** Power plants, especially those using thermal processes, rely heavily on water for cooling. For instance, a coal-fired power plant may consume approximately 1,000 gallons of water per megawatt-hour of electricity produced. This significant water usage directly impacts the water footprint of data ce

In [6]:
df_ws.loc[(df_ws.references.isna()) | (df_ws.references == '[]')]


,prompt,result,references,tokens
426,Write a short literature review on the stateme...,NaN,NaN,NaN
429,Write a short literature review on the stateme...,NaN,NaN,NaN
430,Write a short literature review on the stateme...,NaN,NaN,NaN
431,Write a short literature review on the stateme...,NaN,NaN,NaN
432,Write a short literature review on the stateme...,NaN,NaN,NaN
...,...,...,...,...
544,Write a short literature review on the stateme...,NaN,NaN,NaN
545,Write a short literature review on the stateme...,NaN,NaN,NaN
546,Write a short literature review on the stateme...,NaN,NaN,NaN
547,Write a short literature review on the stateme...,NaN,NaN,NaN


In [11]:
# Reemplazar en df_ws los registros que aparecen en df_ws2
df_ws = df_ws.set_index('prompt')
df_ws2_indexed = df_ws2.set_index('prompt')

df_ws.update(df_ws2_indexed)

df_ws = df_ws.reset_index()
df_ws.shape

(1042, 4)

In [14]:
df_ws.to_csv('../data/results_gpt_web_search_V2.csv', index=False)

In [15]:
import re
from urllib.parse import urlsplit, urlunsplit
import pandas as pd

# Regex general para URLs (http/https), evita capturar paréntesis/llaves/corchetes al final
_URL_RE = re.compile(r'https?://[^\s<>"\']+')

def _clean_url(u: str) -> str:
    """
    Limpia basura común al final: ), ], }, ., ,, ;, :
    y normaliza levemente.
    """
    u = u.strip()

    # Recorta cierres típicos que se pegan en Markdown o puntuación final
    while u and u[-1] in ')]}.,;:':
        u = u[:-1]

    # Opcional: normaliza (sin tocar query/utm)
    # (esto evita cosas raras como espacios u otros, pero es suave)
    parts = urlsplit(u)
    return urlunsplit(parts)

def extract_urls_from_text(text) -> list[str]:
    """
    Devuelve una lista (orden de aparición) con todas las URLs encontradas en el texto.
    Maneja NaN/None.
    """
    if text is None:
        return []
    # Pandas puede traer NaN (float)
    if isinstance(text, float) and pd.isna(text):
        return []

    s = str(text)

    urls = []
    for m in _URL_RE.finditer(s):
        urls.append(_clean_url(m.group(0)))

    # De-duplicar manteniendo orden
    seen = set()
    out = []
    for u in urls:
        if u and u not in seen:
            seen.add(u)
            out.append(u)
    return out

# --- Ejemplo de uso con DataFrame ---
# df["urls"] tendrá una lista por fila con todas las URLs encontradas en df["texto"]
# df["urls"] = df["texto"].apply(extract_urls_from_text)

# Si quieres una sola lista con todas las URLs del dataframe (aplanada):
# all_urls = [u for lst in df["urls"] for u in lst]

In [16]:
df["urls_clean"] = df["result"].apply(extract_urls_from_text)

In [25]:
df_ws['urls_clean'] = df_ws['result'].apply(extract_urls_from_text)

In [17]:
df

,prompt,result,references,tokens,urls_clean
0,"I want to write an article about: ""Common fair...",Several studies demonstrate the mathematical i...,['https://jmlr.org/beta/papers/v24/22-1511.htm...,NaN,[https://jmlr.org/beta/papers/v24/22-1511.html...
1,"I want to write an article about: ""Machine Lea...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN,[https://www.forbes.com/sites/aparnadhinakaran...
2,"I want to write an article about: ""Evaluation ...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN,[https://arxiv.org/abs/2006.09663?utm_source=o...
3,"I want to write an article about: ""Benchmark c...",Several studies highlight how US-centric bench...,['https://davidnowak.me/why-your-ai-benchmarks...,NaN,[https://davidnowak.me/why-your-ai-benchmarks-...
4,"I want to write an article about: ""Word embedd...",Several studies have demonstrated that word em...,['https://arxiv.org/abs/1903.03862?utm_source=...,NaN,[https://arxiv.org/abs/1903.03862?utm_source=o...
...,...,...,...,...,...
2130,"Define ""Positive bias"". Include references.",**Positive Bias** refers to the tendency to fa...,[],NaN,[https://www.psychologytoday.com/us/basics/cog...
2131,"Define ""Governance in AI"". Include references.","""Governance in AI"" refers to the frameworks, p...",[],NaN,[https://www.oecd.org/going-digital/ai/princip...
2132,"Define ""Intrinsic hallucinations"". Include ref...",**Intrinsic hallucinations** are sensory perce...,[],NaN,[https://www.ncbi.nlm.nih.gov/pmc/articles/PMC...
2133,"Define ""Extrinsic hallucinations"". Include ref...",Extrinsic hallucinations refer to sensory perc...,[],NaN,[https://www.ncbi.nlm.nih.gov/pmc/articles/PMC...


In [ ]:
df['urls_clean'].iloc[56]

In [ ]:
# TODO: Revisar las etiquetas unknown y ver cómo depuramos que sea peer reviewd o no.
# MISTRAL : Mirar referencias falsas por medio de google scholar (podemos comparar con algún cálculo de diferencia
# la cita en APA con la referencia que arroja el modelo - esto para los que aparecen en scholar pero puede pasar que
# haya referencias que ni el título exista)

In [ ]:
from urllib.parse import urlsplit

def base_url(url: str) -> str:
    """scheme + netloc. Ej: https://www.economist.com"""
    if not url:
        return ""
    p = urlsplit(str(url))
    if not p.scheme or not p.netloc:
        return ""
    return f"{p.scheme}://{p.netloc}"

def extract_baseurl_label_pairs_from_lists(
    df: pd.DataFrame,
    urls_col: str = "urls_clean",
    labels_col: str = "labels",
) -> pd.DataFrame:
    """
    df[urls_col]: lista de urls por fila
    df[labels_col]: lista de labels por fila (misma longitud idealmente)
    Devuelve DF largo con columnas: base_url, label
    """
    rows = []

    for urls, labels in zip(df[urls_col], df[labels_col]):
        if not isinstance(urls, list) or not isinstance(labels, list):
            continue

        # zip tolerante: corta al mínimo si vienen desalineadas
        for u, lab in zip(urls, labels):
            b = base_url(u)
            if b:
                rows.append({"base_url": b, "label": lab if lab is not None else "unknown"})

    return pd.DataFrame(rows, columns=["base_url", "label"])

def unique_labels_by_baseurl(pairs_df: pd.DataFrame) -> pd.DataFrame:
    """
    Agrupa por base_url y devuelve labels únicos por base_url.
    """
    if pairs_df.empty:
        return pd.DataFrame(columns=["base_url", "unique_labels", "n_unique_labels"])

    out = (
        pairs_df.groupby("base_url")["label"]
        .agg(lambda s: sorted(set(s)))
        .reset_index(name="unique_labels")
    )
    out["n_unique_labels"] = out["unique_labels"].apply(len)
    return out



In [18]:
import ast
import pandas as pd

def to_list_safe(x):
    """
    Convierte:
    - NaN/None -> []
    - lista -> lista
    - string tipo "['a','b']" -> lista
    - string 'nan' / '' -> []
    - cualquier otro -> []
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s == "" or s.lower() == "nan":
            return []
        # si parece lista serializada
        if s.startswith("[") and s.endswith("]"):
            try:
                return ast.literal_eval(s)
            except (ValueError, SyntaxError):
                return []
        return []
    return []

# Ejemplo:
# df["urls_clean"] = df["urls_clean"].apply(to_list_safe)

In [20]:
import requests
import pandas as pd

def url_exists(url, timeout: int = 15, allow_redirects: bool = True,
               accept_3xx: bool = False, max_bytes: int = 2048) -> dict:
    """
    Valida acceso real a la URL completa (GET, no solo HEAD).
    """
    # Normaliza / filtra NaN y valores raros
    if url is None:
        return {"url": url, "exists": False, "status_code": None, "final_url": None, "error": "empty"}
    if isinstance(url, float) and pd.isna(url):
        return {"url": None, "exists": False, "status_code": None, "final_url": None, "error": "empty"}
    if not isinstance(url, str):
        url = str(url)

    url = url.strip()
    if url == "" or url.lower() == "nan":
        return {"url": url, "exists": False, "status_code": None, "final_url": None, "error": "empty"}

    try:
        with requests.get(
            url,
            timeout=timeout,
            allow_redirects=allow_redirects,
            headers={"User-Agent": "url-validator/1.0"},
            stream=True,
        ) as r:
            status = r.status_code

            # Lee un poquito para confirmar acceso sin bajar todo
            try:
                for chunk in r.iter_content(chunk_size=max_bytes):
                    if chunk:
                        break
            except Exception:
                pass

            if 200 <= status < 300:
                exists = True
            elif accept_3xx and 300 <= status < 400:
                exists = True
            elif status in [402, 403, 406, 502, 503, 405]: 
                exists = True
            elif status is None:
                exists = None
            else:
                exists = False

            return {
                "url": url,
                "exists": exists,
                "status_code": status,
                "final_url": str(r.url),
                "error": None,
            }

    except requests.exceptions.RequestException as e:
        return {
            "url": url,
            "exists": False,
            "status_code": None,
            "final_url": None,
            "error": f"{type(e).__name__}: {e}",
        }

def url_exists_list(urls, **kwargs) -> list[dict]:
    """
    Aplica url_exists a una lista, tolerante a NaN y strings serializados.
    """
    urls = to_list_safe(urls)  # <- usa la función de arriba
    return [url_exists(u, **kwargs) for u in urls if u not in (None, "")]

In [21]:
ress_df = df["urls_clean"].progress_apply(lambda lst: url_exists_list(lst, timeout=15))

100%|██████████| 2135/2135 [1:39:08<00:00,  2.79s/it]  


In [23]:
df_ws

,prompt,result,references,tokens
0,"Explain the following statement: ""Common fairn...","The statement ""Common fairness definitions are...",['https://algorithmaudit.eu/knowledge-platform...,NaN
1,"Explain the following statement: ""Machine Lear...",Machine learning models can exhibit bias even ...,['https://www.forbes.com/sites/aparnadhinakara...,NaN
2,"Explain the following statement: ""Evaluation a...",Evaluating and mitigating fairness solely thro...,['https://arxiv.org/abs/2006.09663?utm_source=...,NaN
3,"Explain the following statement: ""Benchmark ch...",Benchmark choices for Generative AI (GenAI) mo...,['https://www.tomsguide.com/ai/ai-safety-tests...,NaN
4,"Explain the following statement: ""Word embeddi...","Word embeddings, which are vector representati...",['https://pubmed.ncbi.nlm.nih.gov/33400629/?ut...,NaN
...,...,...,...,...
1037,"Define ""Positive bias"". Include references.","A positive bias, also known as positivity bias...",['https://psychology.tips/positive-bias/?utm_s...,NaN
1038,"Define ""Governance in AI"". Include references.",AI governance refers to the framework of polic...,['https://www.unesco.org/en/query-list/a/ai-go...,NaN
1039,"Define ""Intrinsic hallucinations"". Include ref...","In the context of artificial intelligence, par...",['https://urca.foundation/hallucination-in-ai/...,NaN
1040,"Define ""Extrinsic hallucinations"". Include ref...","In the context of artificial intelligence, ""ex...",['https://www.emergentmind.com/topics/hallucin...,NaN


In [26]:
ress_df_ws = df_ws["urls_clean"].progress_apply(lambda lst: url_exists_list(lst, timeout=15))

100%|██████████| 1042/1042 [49:39<00:00,  2.86s/it] 


In [28]:
ress_df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 2135 entries, 0 to 2134
Series name: urls_clean
Non-Null Count  Dtype 
--------------  ----- 
2135 non-null   object
dtypes: object(1)
memory usage: 16.8+ KB


In [30]:
df['status_check'] =ress_df
df_ws['status_check'] = ress_df_ws

In [32]:
import ast
def extract_status_codes(url_checks):
    """
    Dado url_checks (lista de dicts), devuelve lista de status_code.
    Maneja NaN/None.
    """
    if url_checks is None:
        return []
    if isinstance(url_checks, float) and pd.isna(url_checks):
        return []
    if isinstance(url_checks, str):
        try:
            url_checks = ast.literal_eval(url_checks)
        except (ValueError, SyntaxError):
            return []

    if not isinstance(url_checks, list):
        return []

    status_codes = []
    for check in url_checks:
        if isinstance(check, dict) and "status_code" in check:
            status_codes.append(check["exists"])
    return status_codes

In [33]:
df['status_codes'] = df['status_check'].apply(extract_status_codes)

In [34]:
df_ws['status_codes'] = df_ws['status_check'].apply(extract_status_codes)

In [35]:
def get_total(lista):
    return lista.count(False), lista.count(True)

In [36]:
df['Total_False'], df['Total_True'] = zip(*df['status_codes'].apply(get_total))

In [37]:
df_ws['Total_False'], df_ws['Total_True'] = zip(*df_ws['status_codes'].apply(get_total))

In [38]:
df.Total_False.sum(), df.Total_True.sum(),  df.Total_False.sum() + df.Total_True.sum(), df.shape

(np.int64(2134), np.int64(5143), np.int64(7277), (2135, 9))

In [39]:
df_ws.Total_False.sum(), df_ws.Total_True.sum(),  df_ws.Total_False.sum() + df_ws.Total_True.sum(), df_ws.shape

(np.int64(83), np.int64(3617), np.int64(3700), (1042, 9))

In [53]:
import ast

def extract_existing_urls(df: pd.DataFrame, col: str) -> pd.DataFrame:
    """
    Extrae todas las URLs con exists=True de una columna con listas de dicts,
    y retorna un DataFrame donde cada URL es una fila.
    """
    rows = []

    for idx, cell_value in df[col].items():
        if isinstance(cell_value, list):
            records = cell_value
        else:
            try:
                records = ast.literal_eval(cell_value)
            except (ValueError, SyntaxError):
                continue

        for record in records:
            if record.get("exists") is True:
                rows.append({
                    "original_row": idx,        # índice original del df, útil para trazabilidad
                    "url": record.get("url"),
                    "status_code": record.get("status_code"),
                    "final_url": record.get("final_url"),
                    "error": record.get("error"),
                })

    return pd.DataFrame(rows)


In [54]:
# Usar
urls_df = extract_existing_urls(df, "status_check")
urls_df2 = extract_existing_urls(df_ws, "status_check")

In [55]:
urls_df.shape

(5143, 5)

In [56]:
urls_df2.shape

(3617, 5)

In [ ]:
urls_df["url"].to_csv("urls1.txt", index=False, header=False)
urls_df2["url"].to_csv("urls2.txt", index=False, header=False)

In [57]:
from classify_urls import process_urls

urls_list = urls_df["url"].tolist()
results = process_urls(urls_list, workers=8)

classified_df = pd.DataFrame(results)

[██████████████████████████████] 5143/5143  


In [58]:
urls_list2 = urls_df2["url"].tolist()
results2 = process_urls(urls_list2, workers=8)

classified_df2 = pd.DataFrame(results2)

[██████████████████████████████] 3617/3617  


In [59]:
classified_df.peer_reviewed.value_counts(dropna=False)

peer_reviewed
no    3466
sí    1677
Name: count, dtype: int64

In [60]:
classified_df2.peer_reviewed.value_counts(dropna=False)

peer_reviewed
no    2687
sí     930
Name: count, dtype: int64